# TB Portals â€” 01 Â· Build manifest

Day 1. Produces `manifest.csv` with the fixed contract columns: `image_id, image_path, patient_id, country, alp_0_100, cavity`.

- **Real data:** edit `COLUMN_MAP` below to your actual TB Portals reads column names (passed inline â€” no source edit needed).
- **No data yet:** run the *synthetic* cell to exercise the whole stack now.

In [1]:

  import sys, os, subprocess

  REPO_DIR = '/kaggle/working/dl-project-codebase'

  # Delete stale clone (it's on main, not cleaned-repo)
  if os.path.isdir(REPO_DIR):
      subprocess.run(["rm", "-rf", REPO_DIR], check=True)

  subprocess.run([
      "git", "clone", "--depth", "1",
      "--branch", "cleaned-repo",
      "https://github.com/mabdullahi7780/dl-project-codebase.git",
      REPO_DIR
  ], check=True)

  if REPO_DIR not in sys.path:
      sys.path.insert(0, REPO_DIR)

  print("Cloned cleaned-repo. Files present:")
  print(os.listdir(f"{REPO_DIR}/src/data"))

Cloning into '/kaggle/working/dl-project-codebase'...


Cloned cleaned-repo. Files present:
['tbportals.py', 'transforms_qc.py', 'tbportals_dataset.py', 'harmonise.py', 'component4_lung_dataset.py', 'component1_dann_dataset.py', '__init__.py']


Updating files: 100% (428/428), done.


In [2]:
# --- Clone / update the repo (requires Internet enabled in Kaggle) ---
import os, subprocess, sys

REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo ready at", REPO_DIR)

Already up to date.
repo ready at /kaggle/working/dl-project-codebase


In [3]:
import sys, os, pandas as pd

REPO_DIR    = '/kaggle/working/dl-project-codebase'
DATASET_DIR = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs-full'
KAGGLE_EXPORT = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs-full/kaggle_export_full'
WORK        = '/kaggle/working'
sys.path.insert(0, REPO_DIR)
os.makedirs(f'{WORK}/data/processed', exist_ok=True)

MANIFEST = f'{WORK}/data/processed/tbportals_manifest.csv'
print('python', sys.version.split()[0])
print('images at: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs-full/kaggle_export_full/images')


python 3.12.12
images at: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs-full/kaggle_export_full/images


## Load manifest from uploaded PNG dataset

The `tb-portals-cxr-pngs` Kaggle dataset was created locally by running:
```
python scripts/export_kaggle_dataset.py \
    --manifest local_work/data/processed/tbportals_manifest.csv \
    --out-dir  local_work/kaggle_export
```
It contains `images/*.png` (512x512 grayscale) and `manifest.csv` with relative paths.
This cell rewrites `image_path` to absolute Kaggle paths.


In [4]:
import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
# Ensure repo is cloned and on sys.path
if not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mabdullahi7780/dl-project-codebase.git", REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.tbportals import load_manifest, summarize_manifest

# Load pre-built manifest from the uploaded Kaggle dataset
_raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv',
                   dtype={'image_id': str, 'patient_id': str, 'country': str})

# Rewrite relative paths (images/<stem>.png) -> absolute Kaggle paths
_raw['image_path'] = _raw['image_path'].apply(
    lambda p: f"{KAGGLE_EXPORT}/{p}" if isinstance(p, str) else p
)

_raw.to_csv(MANIFEST, index=False)
df = load_manifest(MANIFEST)
print(f'Loaded {len(df)} images')
summarize_manifest(df)


Loaded 16990 images
[tbportals] manifest summary:
              n_images  n_patients  alp_mean  alp_std  cavity_rate
country                                                           
Azerbaijan          25          17     40.00    21.11         0.28
Belarus           2258        1176     17.94    20.85         0.25
Georgia           3592        3588     31.04    20.45         0.61
India               18          18     25.67    20.34         0.33
Kazakhstan        2037        1506     26.35    21.90         0.44
Kyrgyzstan         712         672     25.39    20.60         0.68
Moldova            812         811     40.27    30.21         0.35
Romania            823         498     34.47    23.39         0.57
Senegal              2           2     55.00    63.64         0.00
South Africa       135         100     22.04    26.04         0.42
Ukraine           6576        6534     26.82    22.21         0.36
[tbportals] TOTAL images=16990 patients=14922


,n_images,n_patients,alp_mean,alp_std,cavity_rate
country,,,,,
Azerbaijan,25,17,40.00,21.11,0.28
Belarus,2258,1176,17.94,20.85,0.25
Georgia,3592,3588,31.04,20.45,0.61
India,18,18,25.67,20.34,0.33
Kazakhstan,2037,1506,26.35,21.90,0.44
Kyrgyzstan,712,672,25.39,20.60,0.68
Moldova,812,811,40.27,30.21,0.35
Romania,823,498,34.47,23.39,0.57
Senegal,2,2,55.00,63.64,0.00


## Verify

In [5]:
df = load_manifest(MANIFEST)
summarize_manifest(df)
for c in ['Romania', 'Moldova', 'Kazakhstan']:
    assert c in set(df['country']), f'Held-out country missing: {c}'
print('\nManifest OK ->', MANIFEST)

[tbportals] manifest summary:
              n_images  n_patients  alp_mean  alp_std  cavity_rate
country                                                           
Azerbaijan          25          17     40.00    21.11         0.28
Belarus           2258        1176     17.94    20.85         0.25
Georgia           3592        3588     31.04    20.45         0.61
India               18          18     25.67    20.34         0.33
Kazakhstan        2037        1506     26.35    21.90         0.44
Kyrgyzstan         712         672     25.39    20.60         0.68
Moldova            812         811     40.27    30.21         0.35
Romania            823         498     34.47    23.39         0.57
Senegal              2           2     55.00    63.64         0.00
South Africa       135         100     22.04    26.04         0.42
Ukraine           6576        6534     26.82    22.21         0.36
[tbportals] TOTAL images=16990 patients=14922

Manifest OK -> /kaggle/working/data/processed/tbport